In [40]:
import os
import glob
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict
from daemon_analysis_tools.io.yaml_handler import load_answers_from_yaml
from daemon_analysis_tools.io.csv_handler import _load_csv

In [41]:
csv_read = _load_csv("../../data/metadata/journal_selection.csv")

impact_factor_dict = {}
for journal, impact_factor in zip(csv_read["title"], csv_read["citations_avg"]):
    impact_factor_dict[journal.lower().replace(" ", "_")] = impact_factor

print(impact_factor_dict)
#print(csv_read["title"], csv_read["citations_avg"], csv_read["citations_median"])


[info] CSV loaded ← ../../data/metadata/journal_selection.csv
{'escience': 47.79047619047619, 'applied_catalysis_b_environmental': 39.88368891947695, 'energy_storage_materials': 28.907997169143663, 'nano_energy': 29.69001610305958, 'food_hydrocolloids': 27.77777777777778, 'advanced_powder_materials': 47.19642857142857, 'joule': 43.5125348189415, 'ssrn_electronic_journal': 0.1664711127798376, 'materials_today_proceedings': 5.190393518518518, 'chemical_engineering_journal': 21.943821038152464, 'journal_of_alloys_and_compounds': 10.656837506825806, 'ceramics_international': 8.925697674418604, 'bioactive_materials': 22.83080808080808, 'composites_part_a_applied_science_and_manufacturing': 14.213085764809902, 'materials_science_and_engineering_a': 11.167504630854724, 'superlattices_and_microstructures': 9.12735849056604, 'materials_today_bio': 8.105442176870747, 'accounts_of_chemical_research': 29.07106598984772, 'acs_energy_letters': 27.080745341614907, 'journal_of_the_american_chemical_so

In [42]:
SMALL_SIZE = 8
MEDIUM_SIZE = 10
BIGGER_SIZE = 12
LINE_WIDTH = 1.5
MARKER_SIZE = 6

plt.rc("font", **{"family": "sans-serif", "sans-serif": ["Liberation Sans"]})
plt.rc("font", family="sans-serif", weight="bold", size=MEDIUM_SIZE)

plt.rc("axes", titlesize=MEDIUM_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)

mpl.rcParams["font.weight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
mpl.rcParams["mathtext.default"] = "regular"
mpl.rcParams["lines.linewidth"] = LINE_WIDTH
mpl.rcParams["lines.markersize"] = MARKER_SIZE

In [43]:
# Load all the processed yaml files 
processed_data = load_answers_from_yaml('../../data/processed/all_answers/')


No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
No discrepancies detected.
N

In [45]:
import os
from collections import defaultdict, Counter
import plotly.graph_objects as go
from daemon_analysis_tools.services.scoring import get_question_score, _get_question_type_lookup
from daemon_analysis_tools.io.yaml_base_handler import load_yaml

# Load metadata
data_dir = os.path.join("..", "..", "src/daemon_analysis_tools/metadata")
multiple_choice_scores = load_yaml(os.path.join(data_dir, "question_metadata_score.yaml"))
question_type = load_yaml(os.path.join(data_dir, "question_type.yaml"))
question_number_lookup = _get_question_type_lookup(multiple_choice_scores)

# Collect flow data
question_answer_count = defaultdict(Counter)
answer_score_map = {}

for publisher, publisher_results in processed_data.items():
    for journal, journal_results in publisher_results.items():
        for question, answer in journal_results.items():

            print(question, answer.correct_answer.text)

            question_answer_count[question][answer.correct_answer.text] += 1
            if (question, answer) not in answer_score_map:
                score = get_question_score(
                    question,
                    answer,
                    multiple_choice_scores,
                    question_type,
                    question_number_lookup,
                )
                answer_score_map[(question, answer.correct_answer.text)] = score

# Build node labels
questions = list(question_answer_count.keys())
answers = {ans for counts in question_answer_count.values() for ans in counts}
scores = set(answer_score_map.values())

# Assign node indices
label_list = []
node_map = {}

# Questions
for q in questions:
    label = q.replace("_", " ")
    node_map[("q", q)] = len(label_list)
    label_list.append(label)

# Answers
for a in answers:
    node_map[("a", a)] = len(label_list)
    label_list.append(str(a))

# Scores
for s in scores:
    node_map[("s", s)] = len(label_list)
    label_list.append(f"score {s}")

# Create links: Question → Answer and Answer → Score
source = []
target = []
value = []

# Question to Answer
for q, answers_count in question_answer_count.items():
    for a, count in answers_count.items():
        source.append(node_map[("q", q)])
        target.append(node_map[("a", a)])
        value.append(count)

# Answer to Score
answer_score_count = defaultdict(int)
for q, answers_count in question_answer_count.items():
    for a, count in answers_count.items():
        score = answer_score_map[(q, a)]
        answer_score_count[(a, score)] += count

for (a, s), count in answer_score_count.items():
    source.append(node_map[("a", a)])
    target.append(node_map[("s", s)])
    value.append(count)

# Plot the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=label_list,
        color="skyblue"
    ),
    link=dict(
        source=source,
        target=target,
        value=value
    ))])

fig.update_layout(title_text="Question → Answer → Score Flow", font_size=10)

rdp_exist Research Data Policy (RDP) exists.
data_sharing Data sharing encouraged but optional.
data_fair Public data sharing on a FAIR repository not mentioned in RDP.
data_availability Mentioned in the RDP but optional.
data_citability No mention of DOIs or other persistent identifiers for datasets or codes.
data_timing Required data must be available prior to official publication.
data_sharing_method Data sharing in supplementary material or hosting by journal recommended in RDP.
data_licenses No mention of data/code licenses in RDP.
data_referee Data sharing policy not mentioned in refereeing guidelines.
data_recommended no text.
data_required crystal structures
code_required Code sharing required.
code_reproducibility Journal policies require stating all dependencies and their versions that were used to run computational experiments.
code_versioning Journal policies encourage/recommend a persistent identifier or specifying a version of developed code.
code_quality The journal poli